In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np


In [2]:
# Transformações
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Carregar dataset completo
data_dir = "/home/lobo/github/RepLearningLosses/PlatonicSolidsImages"
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# Organizar índices por classe
class_indices = {cls: [] for cls in range(len(dataset.classes))}
for idx, (_, label) in enumerate(dataset.samples):
    class_indices[label].append(idx)

# Selecionar 30% dos dados para treino
train_indices = []
remaining_indices = []

for cls, indices in class_indices.items():
    np.random.shuffle(indices)  # Embaralhar para evitar viés
    split_idx = int(0.3 * len(indices))  # 30% dos dados dessa classe
    train_indices.extend(indices[:split_idx])
    remaining_indices.extend(indices[split_idx:])  # O restante vai para validação/teste

# Criar conjuntos de treino, validação e teste
train_dataset = Subset(dataset, train_indices)

# Dividir o restante em validação (50%) e teste (50%)
val_size = len(remaining_indices) // 2
val_dataset = Subset(dataset, remaining_indices[:val_size])
test_dataset = Subset(dataset, remaining_indices[val_size:])

# Criar DataLoaders
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f"Treino: {len(train_dataset)} imagens")
print(f"Validação: {len(val_dataset)} imagens")
print(f"Teste: {len(test_dataset)} imagens")


Treino: 1965 imagens
Validação: 2293 imagens
Teste: 2294 imagens


In [3]:
len(dataset)

6552

In [3]:
from collections import Counter

# Obter rótulos originais
all_labels = [label for _, label in dataset.samples]

# Contar classes no conjunto de treino
train_labels = [all_labels[idx] for idx in train_indices]
train_class_counts = Counter(train_labels)

# Contar classes no conjunto de validação
val_labels = [all_labels[idx] for idx in remaining_indices[:val_size]]
val_class_counts = Counter(val_labels)

# Contar classes no conjunto de teste
test_labels = [all_labels[idx] for idx in remaining_indices[val_size:]]
test_class_counts = Counter(test_labels)

# Exibir os resultados
print("\nDistribuição das classes:")
for cls, class_name in enumerate(dataset.classes):
    print(f"Classe '{class_name}':")
    print(f"  ➤ Treino: {train_class_counts.get(cls, 0)} imagens")
    print(f"  ➤ Validação: {val_class_counts.get(cls, 0)} imagens")
    print(f"  ➤ Teste: {test_class_counts.get(cls, 0)} imagens")



Distribuição das classes:
Classe 'test':
  ➤ Treino: 267 imagens
  ➤ Validação: 624 imagens
  ➤ Teste: 0 imagens
Classe 'train':
  ➤ Treino: 1698 imagens
  ➤ Validação: 1669 imagens
  ➤ Teste: 2294 imagens
